# **Automatic Alternate Abbreviation Annotation:** LLM predictions on gene-alias pairs for Alternate Abbreviation alias symbols

This notebook runs the Alternate Abbreviation annotation workflow on a dataset of gene alias symbols. Alias symbols are first filtered using heuristic rules, after which those requiring further review are submitted to the LLM for annotation. Results are cached and stored for downstream analysis. For more information on Alternate Abbreviations and the overall workflow, see [README](alternate_abbreviation_README).

In [1]:
import re
from pathlib import Path
import pickle
import polars as pl
from tqdm.notebook import tqdm
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner
from rapidfuzz.distance import LCSseq
from alt_abbrev_models import AlternateAbbreviationPredictionResult
from alt_abbrev_models import SkipReason
from alt_abbrev_models import AlternateAbbreviationPrompt

In [2]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 150


def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM alternate abbreviation alias annotator

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(AlternateAbbreviationPrompt(version="v1"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

In [ ]:
def should_call_llm(alias_symbol, primary_gene_symbol, gene_name, threshold=0.20):
    """Determine whether an alias symbol should be sent to the LLM for evaluation. These filters determine that the alias symbol is too different from the primary gene symbol to be an alternate abbreviation or has a determining characteristic that defines it as not an alternate abbreviation.

    Applies a series of heuristic filters to the alias symbol
    - checks for excluded prefixes ("HSA-", this is not an alternate abbreviation alias
    because it is an identifier being used as an alias)
    - extra characters in the alias, not in the gene name
    - longest common subsequence (LCS) similarity to the primary gene symbol to be at
    least whatever the threshold is set to

    :param alias_symbol: The alias gene symbol being evaluated
    :param primary_gene_symbol: The official primary gene symbol
    :param gene_name: The full gene name associated with the gene
    :param threshold: Minimum LCS similarity score required to pass filtering
    :return: A tuple containing:
        - bool: Whether the alias should be sent to the LLM
        - float: The computed LCS similarity score
        - str: The reason code for the decision
    """

    def lcs_similarity(s1: str, s2: str) -> float:
        """Calculate the longest common subsequence (LCS) similarity

        :param s1: the first string, alias symbol
        :param s2: the second string, primary gene symbol
        :return: value between 0 and 1
        """
        return LCSseq.normalized_similarity(s1, s2)

    def has_extra_characters(alias: str, gene_name: str) -> int:
        """Denote if the alias symbol has characters not present in the gene name

        :param alias: gene alias symbol
        :param gene_name: HGNC designated official gene name
        :return: Number of extra characters
        """
        norm_alias = re.sub(r"[^A-Z0-9]", "", alias.upper())

        alias_set = set(norm_alias)
        name_set = set(gene_name)

        extra_chars = alias_set - name_set
        return len(extra_chars)

    alias = alias_symbol.upper()
    primary = primary_gene_symbol.upper()
    name = gene_name.upper()

    if alias.startswith("HSA-"):
        return False, 0, SkipReason.HSA_PREFIX

    extra_count = has_extra_characters(alias_symbol, name)

    if extra_count >= 2:
        return False, 0, SkipReason.EXTRA_CHARACTERS

    lcs_score = lcs_similarity(alias, primary)

    if lcs_score < threshold:
        return False, lcs_score, SkipReason.LOW_LCS_SIMILARITY

    return True, lcs_score, None

In [4]:
def get_alt_abbreviation_annotation(
    task_runner: StructuredTaskRunner,
    prompt: AlternateAbbreviationPrompt,
    gene_symbol: str,
    primary_gene_symbol: str,
    gene_name: str,
    hgnc_id: str,
) -> AlternateAbbreviationPredictionResult:
    """LLM prediction whether a gene symbol is an alternate abbreviation of a
    primary HGNC gene symbol

    If the alias symbol does not require LLM evaluation, a result containing the
    skip reason and similarity score is returned. Otherwise, the configured
    prompt is executed and the resulting annotation is returned along with the
    computed similarity score.

    :param task_runner: Structured task runner used to execute the LLM prompt.
    :prompt: Configured prompt instance for alternate abbreviation annotation.
    :param gene_symbol: Candidate alias symbol to evaluate.
    :param primary_gene_symbol: Primary HGNC-approved gene symbol being
        compared against.
    :param gene_name: Full gene name associated with the primary gene symbol.
    :param hgnc_id: HGNC identifier for the primary gene.
    :return: Annotation result containing the LLM prediction (when available),
        similarity score, skip reason, or error information.
    """

    call_llm, score, skip_reason = should_call_llm(
        gene_symbol,
        primary_gene_symbol,
        gene_name,
    )

    if not call_llm:
        return AlternateAbbreviationPredictionResult(
            llm_skip_reason=skip_reason,
            lcs_similarity_score=score,
        )

    payload = prompt.build_payload(
        gene_symbol=gene_symbol,
        primary_gene_symbol=primary_gene_symbol,
        gene_name=gene_name,
        hgnc_id=hgnc_id,
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=AlternateAbbreviationPredictionResult,
        )

        return AlternateAbbreviationPredictionResult(
            llm_annotation=task_result.llm_annotation,
            lcs_similarity_score=score,
        )

    except Exception as e:
        return AlternateAbbreviationPredictionResult(
            error_message=str(e),
            lcs_similarity_score=score,
        )

In [5]:
def run_experiments(df, temperatures, num_runs, prompt_version: str):
    """Run LLM annotation experiments across a range of temperatures and multiple runs per temperature, storing results for analysis.

    :param df: Input dataframe containing gene symbols and associated information.
    :param temperatures: List of temperature values to experiment with.
    :param num_runs: Number of runs to execute for each temperature setting.
    :param prompt_version: Version of the prompt template to use for annotation.
    :return: List of stored runs with LLM outputs and diagnostics for each temperature and run
    """
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = AlternateAbbreviationPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for row in tqdm(
                df.iter_rows(named=True),
                total=df.height,
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                result = get_alt_abbreviation_annotation(
                    task_runner=task_runner,
                    prompt=prompt,
                    gene_symbol=row["gene_symbol"],
                    primary_gene_symbol=row["primary_gene_symbol"],
                    gene_name=row["gene_name"],
                    hgnc_id=row["HGNC_ID"],
                )

                if result.error_message:
                    raise RuntimeError(
                        f"""
                        Error in temp={temp}, run={run_idx + 1}
                        gene_symbol={row["gene_symbol"]}
                        primary_gene_symbol={row["primary_gene_symbol"]}
                        HGNC_ID={row["HGNC_ID"]}
                        error={result.error_message}
                        """
                    )

                results.append(
                    {
                        "llm_annotation": result.llm_annotation,
                        "llm_skip_reason": result.llm_skip_reason,
                        "error_message": result.error_message,
                        "lcs_similarity_score": result.lcs_similarity_score,
                        "gt": row["alternate_abbreviation_status"],
                    }
                )
            print("prompt_version being stored:", prompt_version)

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

In [6]:
def build_experiment_key(sample_name, prompt_version, temperatures, num_runs):
    """Build a unique key for identifying an experiment configuration based on sample name, prompt version, temperatures, and number of runs.

    :param sample_name: Name of the sample or dataset being used.
    :param prompt_version: Version of the prompt template used in the experiment.
    :param temperatures: List of temperature values used in the experiment.
    :param num_runs: Number of runs executed for each temperature setting.
    :return: A string key uniquely identifying the experiment configuration.
    """
    temp_str = "-".join(str(t).replace(".", "p") for t in temperatures)
    return f"{sample_name}_p{prompt_version}_t{temp_str}_agg{num_runs}"

In [7]:
ALT_ABBREV_ROOT = Path.cwd().resolve()
ALT_ABBREV_OUTPUT_PATH = ALT_ABBREV_ROOT / "output"

## A test is recommended before using resources to run full dataset

In [8]:
RUN_SUBSET = False

### Load dataset with gene-alias pairs manually curated for Alternate Abbreviation alias symbols

In [9]:
df = pl.read_excel(
    ALT_ABBREV_OUTPUT_PATH / "alt_abbrev_annotation_manually_annotated_df.xlsx"
)

### Create a truncated version of the dataset to test

In [10]:
if RUN_SUBSET:
    test_df = df.head(30)
    SAMPLE_PATH = Path(
        ALT_ABBREV_OUTPUT_PATH
        / "subset_alt_abbrev_annotation_manually_annotated_df.xlsx"
    )
    test_df.write_excel(SAMPLE_PATH)
    df = test_df
else:
    SAMPLE_PATH = Path(
        ALT_ABBREV_OUTPUT_PATH / "alt_abbrev_annotation_manually_annotated_df.xlsx"
    )

## Run LLM with gene symbols, name, and prompt

In [11]:
TEMPERATURES = [0.0]
NUM_RUNS = 3
PROMPT_VERSION = "v1"

sample_name = SAMPLE_PATH.stem
temp_str = "-".join(str(t).replace(".", "p") for t in TEMPERATURES)

experiment_key = build_experiment_key(
    sample_name,
    PROMPT_VERSION,
    TEMPERATURES,
    NUM_RUNS,
)

stored_runs_path = (
    ALT_ABBREV_OUTPUT_PATH / "llm_runs" / f"stored_runs_{experiment_key}.pkl"
)

metadata = {
    "sample_path": str(SAMPLE_PATH),
    "sample_name": sample_name,
    "prompt_version": PROMPT_VERSION,
    "temperatures": TEMPERATURES,
    "num_runs": NUM_RUNS,
}

if stored_runs_path.exists():
    with open(stored_runs_path, "rb") as f:
        saved_data = pickle.load(f)

    if saved_data.get("metadata") == metadata:
        print(f"Loading {stored_runs_path}")
        stored_runs = saved_data["runs"]

    else:
        print(f"Metadata mismatch in {stored_runs_path}. Re-running experiments.")

        stored_runs = run_experiments(df, TEMPERATURES, NUM_RUNS, PROMPT_VERSION)

        with open(stored_runs_path, "wb") as f:
            pickle.dump(
                {
                    "metadata": metadata,
                    "runs": stored_runs,
                },
                f,
            )

else:
    print("Running experiments...")

    stored_runs = run_experiments(df, TEMPERATURES, NUM_RUNS, PROMPT_VERSION)

    with open(stored_runs_path, "wb") as f:
        pickle.dump(
            {
                "metadata": metadata,
                "runs": stored_runs,
            },
            f,
        )

Loading /Users/rsaxs014/Desktop/gene-harmony-analysis/analysis/automatic_alternate_abbreviation_annotation/output/llm_runs/stored_runs_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3.pkl


## Summarizes the conditions of the runs stored

In [12]:
sample_names = set()
all_runs = []

for path in (ALT_ABBREV_OUTPUT_PATH / "llm_runs").glob("stored_runs_*.pkl"):
    sample_name = path.stem.replace("stored_runs_", "")
    sample_names.add(sample_name)

    with open(path, "rb") as f:
        data = pickle.load(f)

    for run in data["runs"]:
        run["sample_name"] = sample_name
        all_runs.append(run)

unique_runs = {
    (run["sample_name"], run["prompt_version"], run["temperature"], run["run_idx"])
    for run in all_runs
}

print(f"Files used for runs:{sorted(sample_names)}")
print(f"Number of unique runs: {len(unique_runs)}")
print(f"Unique runs: {unique_runs}")

Files used for runs:['alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3']
Number of unique runs: 6
Unique runs: {('subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 2), ('alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 2), ('subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 1), ('subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 0), ('alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 1), ('alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 0)}


In [13]:
df.write_parquet(stored_runs_path.with_suffix(".parquet"))